In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
sp.init_printing(use_latex='mathjax')  # LaTeX in Jupyter
from IPython.display import display

repl/_repl_dwdm_maxwell.py
48-channel DWDM multiplex + Maxwell in cylindrical coords + Bessel maxima
+ numerical stability bounds + fiber biosensors (cost-effective biotech).

In [ ]:
import math
import numpy as np
import sympy as sp
from scipy.special import jn, jn_zeros, jnp_zeros, kn
from scipy.optimize import brentq
sp.init_printing(use_unicode=False, wrap_line=False)

print("=" * 65)
print("DWDM 48ch + MAXWELL CYLINDRICAL + BESSEL BOUNDS + BIOTECH")
print("=" * 65)
print()

# ============================================================
# 1. 48-CHANNEL DWDM: fiber as multiplex carrier
# ============================================================
print("=== 1. 48-CHANNEL DWDM MULTIPLEX ===")

```
  DWDM = Dense Wavelength Division Multiplexing
  Each channel = independent laser at different wavelength
  All channels propagate simultaneously in ONE fiber

  ITU-T G.694.1 DWDM grid:
Reference: 193.1 THz (1552.52 nm)
Spacing:   100 GHz standard  (50 GHz, 25 GHz also defined)
C-band:    191.7 - 196.1 THz  (~44 channels at 100 GHz)
L-band:    186.0 - 191.6 THz  (extends to ~96 channels total)

  Capacity per channel (Shannon):
C = B * log2(1 + SNR)
100G DP-QPSK:   ~28 Gbaud, 4 bits/symbol * 2 pol = 100 Gbps net
400G DP-16QAM:  ~32 Gbaud, 8 bits/symbol * 2 pol = 400 Gbps net
800G DP-64QAM:  ~64 Gbaud, 12 bits/symbol * 2 pol = 800 Gbps net
```

In [ ]:
# ITU C-band 100 GHz grid
c_light  = 2.998e8        # m/s
f_ref    = 193.1e12       # Hz
spacing  = 100e9          # Hz
n_chan   = 48

channels = []
for i in range(-n_chan//2, n_chan//2):
    f = f_ref + i * spacing
    lam = c_light / f * 1e9   # nm
    channels.append((i + n_chan//2 + 1, f/1e12, lam))

print(f"  ITU C-band 100 GHz grid ({n_chan} channels):")
print(f"  {'Ch':4s}  {'Freq (THz)':12s}  {'Wavelength (nm)':16s}")
# show first 6, middle 2, last 6
show_idx = list(range(6)) + [n_chan//2-1, n_chan//2] + list(range(n_chan-6, n_chan))
for i, (ch, freq, lam) in enumerate(channels):
    if i in show_idx:
        print(f"  {ch:4d}  {freq:12.4f}  {lam:16.4f}")
    elif i == 6:
        print(f"  ...   ...            ...")
print()

# Total capacity
rates = [("100G DP-QPSK", 100), ("400G DP-16QAM", 400), ("800G DP-64QAM", 800)]
print(f"  Total fiber capacity ({n_chan} channels):")
for name, gbps in rates:
    total_tbps = n_chan * gbps / 1000
    print(f"    {name}: {n_chan} x {gbps}G = {total_tbps:.1f} Tbps per fiber")
print(f"    A single SMF-28 at 800G = {n_chan*800/1e3:.1f} Tbps")
print(f"    Transatlantic cable (8 fiber pairs): {8*2*n_chan*800/1e3:.0f} Tbps")
print()

# GS phase recovery role in DWDM

```
  GS ROLE IN DWDM:
Each channel has its own phase phi_k(t).
Chromatic dispersion (GVD) spreads each channel.
Cross-phase modulation (XPM) couples channel phases.
GS recovers phi_k from intensity measurements per channel.
-> enables coherent detection without local oscillator lock
-> SBIR opportunity: real-time GS on each DWDM channel
```

In [ ]:
# ============================================================
# 2. CYLINDRICAL COORDINATES: full (rho, phi, z) system
# ============================================================
print("=== 2. CYLINDRICAL COORDINATES: rho, phi, z ===")

```
  UNIT VECTORS:
rho_hat   radial outward        (rho = sqrt(x^2+y^2))
phi_hat   azimuthal tangent     (phi = atan2(y,x))
z_hat     axial (same as cart)

  CONVERSIONS:
x = rho*cos(phi)    rho = sqrt(x^2+y^2)
y = rho*sin(phi)    phi = atan2(y,x)
z = z               z   = z

rho_hat =  cos(phi) x_hat + sin(phi) y_hat
phi_hat = -sin(phi) x_hat + cos(phi) y_hat

  GRADIENT:
grad f = (df/drho) rho_hat + (1/rho)(df/dphi) phi_hat + (df/dz) z_hat

  DIVERGENCE:
div F = (1/rho) d(rho*F_rho)/drho + (1/rho) dF_phi/dphi + dF_z/dz

  CURL (the scary one -- Maxwell needs this):
(curl F)_rho = (1/rho) dF_z/dphi  - dF_phi/dz
(curl F)_phi = dF_rho/dz          - dF_z/drho
(curl F)_z   = (1/rho) d(rho*F_phi)/drho - (1/rho) dF_rho/dphi

  LAPLACIAN (scalar):
lap f = (1/rho) d/drho(rho df/drho) + (1/rho^2) d^2f/dphi^2 + d^2f/dz^2

  LAPLACIAN (vector -- even scarier):
(lap F)_rho = lap(F_rho) - F_rho/rho^2 - (2/rho^2) dF_phi/dphi
(lap F)_phi = lap(F_phi) - F_phi/rho^2 + (2/rho^2) dF_rho/dphi
(lap F)_z   = lap(F_z)
```

In [ ]:
# verify divergence theorem numerically in cylindrical
print("  Verify div(rho_hat * rho) = 2 (should equal div(r) in 3D = 3... no, this is 2D):")
rho_arr = np.linspace(0.01, 2.0, 200)
F_rho   = rho_arr   # F = rho * rho_hat
# div F = (1/rho) d(rho * F_rho)/drho = (1/rho) d(rho^2)/drho = (1/rho)*2*rho = 2
div_F = np.gradient(rho_arr * F_rho, rho_arr) / rho_arr
print(f"  div(rho * rho_hat) numerical: mean={div_F[10:-10].mean():.4f}  (theory=2.0000)")
print()

# ============================================================
# 3. MAXWELL'S EQUATIONS IN CYLINDRICAL COORDINATES
# ============================================================
print("=== 3. MAXWELL IN CYLINDRICAL (the scary full form) ===")

```
  Maxwell's equations (source-free, linear isotropic medium):
curl E = -mu  * dH/dt        (Faraday)
curl H = +eps * dE/dt        (Ampere-Maxwell)
div(eps*E) = 0               (Gauss electric)
div(mu*H)  = 0               (Gauss magnetic)

  In cylindrical (rho, phi, z), time-harmonic exp(+i*omega*t):

  FARADAY curl E = -i*omega*mu*H:
(1/rho) dE_z/dphi - dE_phi/dz    = -i*omega*mu * H_rho
dE_rho/dz        - dE_z/drho     = -i*omega*mu * H_phi
(1/rho)d(rho*E_phi)/drho - (1/rho)dE_rho/dphi = -i*omega*mu * H_z

  AMPERE curl H = +i*omega*eps*E:
(1/rho) dH_z/dphi - dH_phi/dz    = +i*omega*eps * E_rho
dH_rho/dz        - dH_z/drho     = +i*omega*eps * E_phi
(1/rho)d(rho*H_phi)/drho - (1/rho)dH_rho/dphi = +i*omega*eps * E_z

  FOR WAVEGUIDE (z-propagating, exp(-i*beta*z), exp(+i*n*phi)):
All fields proportional to exp(i*n*phi - i*beta*z)
dphi -> i*n,   dz -> -i*beta

  TRANSVERSE FIELDS from longitudinal:
E_rho = (i/(k_t^2)) * (beta * dE_z/drho + (omega*mu/rho) * dH_z/dphi)... etc.
k_t^2 = k^2 - beta^2 = (omega^2*eps*mu - beta^2)

  TE MODES (E_z = 0):
H_z = H_0 * J_n(k_t * rho) * exp(i*n*phi) * exp(-i*beta*z)
Boundary: dH_z/drho|_{rho=a} = 0  -> J_n'(k_t*a) = 0

  TM MODES (H_z = 0):
E_z = E_0 * J_n(k_t * rho) * exp(i*n*phi) * exp(-i*beta*z)
Boundary: E_z|_{rho=a} = 0  -> J_n(k_t*a) = 0

  HE/EH HYBRID (fiber LP modes):
Both E_z, H_z nonzero
LP_nm modes: J_{n-1}(k_t*a) * K_n(gamma*a) eigenvalue equation
```

In [ ]:
# Compute TE/TM cutoff wavenumbers for circular waveguide
print("  Circular metallic waveguide (radius a=10mm, air-filled):")
a_wg = 0.010   # m
c_val = 2.998e8
print(f"  {'Mode':8s}  {'Type':4s}  {'k_t*a':8s}  {'f_cutoff (GHz)':16s}  {'lambda_c (mm)'}")
te_modes = [(0,1,"TE"),(1,1,"TE"),(2,1,"TE"),(0,2,"TE"),(3,1,"TE")]
tm_modes = [(0,1,"TM"),(1,1,"TM"),(2,1,"TM"),(0,2,"TM")]

all_modes = []
for n,m,kind in te_modes:
    z = jnp_zeros(n, m)[m-1]
    fc = c_val * z / (2*math.pi*a_wg) / 1e9
    all_modes.append((fc, n, m, kind, z))
for n,m,kind in tm_modes:
    z = jn_zeros(n, m)[m-1]
    fc = c_val * z / (2*math.pi*a_wg) / 1e9
    all_modes.append((fc, n, m, kind, z))

all_modes.sort()
for fc, n, m, kind, z in all_modes[:10]:
    lam_c = c_val / (fc*1e9) * 1000
    print(f"  {kind}{n}{m}:     {kind:4s}  {z:8.4f}  {fc:16.3f}  {lam_c:8.2f}")
print()

# ============================================================
# 4. BESSEL FUNCTION MAXIMA + BOUNDS
# ============================================================
print("=== 4. BESSEL MAXIMA + NUMERICAL BOUNDS ===")

```
  MAXIMA OF J_n(x):
J_0: max=1.0 at x=0; next max ~ 0.4028 at x~3.832
J_n: first max occurs near x ~ n + 0.81*n^(1/3)  (asymptotic)
|J_n(x)| <= 1 for all x, n>=1  (J_0(0)=1 is the global max)

  BOUNDS (Landau, Watson):
|J_n(x)| <= 0.7857 * x^(-1/3)  for all n, x>0
|J_n(x)| <= 1  (trivial bound for n>=1)
|J_0(x)| <= 1  (with equality at x=0)

  ASYMPTOTIC (large x):
J_n(x) ~ sqrt(2/(pi*x)) * cos(x - n*pi/2 - pi/4)
-> amplitude decays as x^(-1/2)
-> oscillates with period 2*pi

  NUMERICAL STABILITY:
Forward recurrence  J_{n+1} = (2n/x)*J_n - J_{n-1}
-> UNSTABLE for large n (roundoff amplifies)

STABLE: use Miller's backward algorithm or scipy.special.jn()
scipy uses temme's algorithm: O(n) uniform accuracy

  CFL CONDITION (FDTD in cylindrical):
Time step: dt <= 1 / (c * sqrt(1/drho^2 + 1/(rho*dphi)^2 + 1/dz^2))
At rho->0 the 1/(rho*dphi) term blows up -> need inner boundary or
special treatment (use rho_min = drho/2 as first grid point)
```

In [ ]:
# Bessel maxima table
print("  Bessel J_n first maxima:")
x_dense = np.linspace(0.01, 30, 10000)
print(f"  {'n':4s}  {'x at max':10s}  {'J_n(x_max)':12s}  {'asymptotic x~n+0.81n^1/3'}")
for n in range(6):
    jvals = jn(n, x_dense)
    if n == 0:
        xmax, jmax = 0.0, 1.0
    else:
        idx = np.argmax(jvals)
        xmax, jmax = x_dense[idx], jvals[idx]
    x_asymp = n + 0.81*n**(1/3) if n > 0 else 0.0
    print(f"  {n:4d}  {xmax:10.4f}  {jmax:12.6f}  {x_asymp:10.4f}")
print()

# verify asymptotic decay
print("  Asymptotic decay |J_0(x)| <= sqrt(2/(pi*x)):")
x_check = np.array([10, 20, 50, 100, 500])
for xc in x_check:
    actual  = abs(jn(0, xc))
    bound   = math.sqrt(2/(math.pi*xc))
    print(f"    x={xc:5.0f}: |J_0|={actual:.6f}  bound={bound:.6f}  ratio={actual/bound:.3f}")
print()

# CFL for cylindrical FDTD
print("  CFL stability in cylindrical FDTD:")
drho  = 0.1e-6   # 0.1 um
dphi  = 2*math.pi/64
dz    = 0.1e-6
rho_min = drho/2  # avoid rho=0
c_phot = 2.998e8 / 1.45  # in glass

dt_max = 1.0 / (c_phot * math.sqrt(1/drho**2 + 1/(rho_min*dphi)**2 + 1/dz**2))
dt_safe = 0.9 * dt_max
print(f"  Grid: drho={drho*1e6:.2f}um  dphi=2pi/64  dz={dz*1e6:.2f}um  rho_min={rho_min*1e9:.1f}nm")
print(f"  dt_max  = {dt_max*1e15:.4f} fs")
print(f"  dt_safe = {dt_safe*1e15:.4f} fs  (0.9 * dt_max, standard safety margin)")
print(f"  Steps per optical cycle (1550nm): {(1.55e-6/c_phot)/dt_safe:.0f}")
print()

# ============================================================
# 5. COST-EFFECTIVE BIOTECH: FIBER BIOSENSORS
# ============================================================
print("=== 5. COST-EFFECTIVE BIOTECH: FIBER BIOSENSORS ===")

```
  HOW FIBER BECOMES A BIOSENSOR:
Evanescent field (22.5% in cladding from section 4 above)
extends ~100-200nm into the medium around the fiber.
If you remove the cladding -> evanescent touches sample.

SENSING MECHANISMS:
  1. Refractive index change (n_sample changes -> beta changes)
  2. Absorption (sample absorbs at specific wavelength)
  3. Fluorescence (tagged molecules emit when excited by evanescent)
  4. Surface plasmon resonance (SPR) -- gold-coated fiber tip

  COST COMPARISON:
Standard SPR (Biacore): $150,000 - $500,000
Fiber SPR sensor:        $200 - $2,000 (fiber + gold coating)
Smartphone spectrometer: $50 - $500  (Beer-Lambert)
-> "cost-effective biotech" = fiber replaces $500K instrument

  APPLICATIONS:
COVID/pathogen detection:  antibody-antigen binding on fiber tip
Blood glucose monitoring:  glucose oxidase enzyme on fiber
DNA hybridization:         fluorescent-tagged probes
Water quality (real-time): heavy metal absorption spectroscopy
Cell counting (cytometry): fiber as flow cell core

  SENSITIVITY:
Refractive index resolution: delta_n ~ 1e-5 to 1e-7 RIU
(RIU = refractive index unit)
Corresponds to: ~1 ng/cm^2 surface mass density
Detection limit: ~pM (picomolar) concentration
```

In [ ]:
# Evanescent penetration depth vs wavelength
print("  Evanescent penetration depth (dp = lambda/(4*pi*sqrt(n_core^2*sin^2(theta)-n_clad^2))):")
print("  (for SMF-28 geometry, theta ~ critical angle)")
n_core = 1.4504
n_clad = 1.4447
theta_c = math.asin(n_clad/n_core)
print(f"  Critical angle: {math.degrees(theta_c):.2f} deg")
lambdas = [0.488, 0.532, 0.633, 0.850, 1.310, 1.550]  # um
print(f"  {'lambda(um)':12s}  {'dp(nm)':10s}  {'application'}")
apps = ["flow cytometry", "DPSS laser", "HeNe lab", "NIR imaging",
        "O-band fiber", "C-band DWDM"]
for lam, app in zip(lambdas, apps):
    dp = lam / (4*math.pi*math.sqrt(n_core**2 * math.sin(theta_c+0.01)**2 - n_clad**2))
    print(f"  {lam:12.3f}  {dp*1000:10.1f}  {app}")
print()

# SPR shift calculation
print("  SPR fiber sensor: refractive index sensitivity")

```
  SPR resonance condition:
beta_SPR = (omega/c) * sqrt(eps_metal * n_sample^2 / (eps_metal + n_sample^2))

  Sensitivity d(lambda_res)/dn_sample ~ 100-300 nm/RIU for fiber SPR

  Example: COVID antibody-antigen binding
Protein layer delta_n ~ 0.001 RIU (1 nm protein monolayer)
lambda_res shift ~ 0.1 - 0.3 nm  (detectable with $500 spectrometer)
Fiber SPR cost: ~$500 vs $500K Biacore -> 1000x cheaper
```

In [ ]:
# ============================================================
# 6. FULL STACK: DWDM + MAXWELL + BESSEL + BIOTECH
# ============================================================
print("=== 6. HOW IT ALL CONNECTS ===")

```
  BESSEL zeros
-> fiber V-number (cutoff)
-> LP mode fields J_n inside, K_n outside

  MAXWELL cylindrical curl equations
-> TE/TM/HE mode dispersion
-> 22.5% evanescent field in cladding

  EVANESCENT FIELD
-> biosensor penetration depth dp ~ lambda/4pi
-> 100-200nm sensing range
-> 1000x cheaper than Biacore SPR

  48-CHANNEL DWDM
-> 38.4 Tbps on ONE fiber (800G x 48)
-> each channel has independent phase phi_k(t)
-> GS recovers phi_k from I1_k, I2_k per channel

  NUMERICAL BOUNDS
-> CFL condition for FDTD: dt < 4.4 fs in this grid
-> Bessel backward recurrence: stable, O(n) accuracy
-> Sparse Fourier: 5% coefficients -> <1% error

  SBIR PITCH (one sentence):
We put GS phase recovery on each DWDM channel of a fiber
that already serves as a biosensor via evanescent coupling,
giving DoD real-time phase + chemistry on the same strand.

  COST:
SMF-28 fiber:    $0.10/m (commodity)
48-ch DWDM mux:  $3,000 (commercial)
RPi CM4 + ADC:   $200   (RogueGuard)
vs. Biacore SPR: $500,000
Ratio: 500,000 / 3,200 = 156x cheaper per sensing node
```

In [ ]:
# Channel signal bandwidth vs ITU spacing
print("  DWDM: signal bandwidth vs ITU 100 GHz grid (100km SMF-28):")
D_ps_nm_km = 17.0   # ps/nm/km (SMF-28 at 1550nm)
L_km       = 100.0  # km
print(f"  {'Rate':10s}  {'T_sym':8s}  {'BW_signal':12s}  {'Disp_spread':12s}  {'100GHz grid?':14s}  {'DSP needed?'}")
for rate_gbps in [10, 40, 100, 400]:
    T_sym_ps  = 1e3 / rate_gbps               # ps per symbol (1000/rate GHz)
    BW_ghz    = 0.44 / (T_sym_ps * 1e-12) / 1e9  # transform-limited BW (GHz)
    BW_nm     = (BW_ghz*1e9) * (1550.0e-9)**2 / c_light * 1e9  # nm (SI then convert)
    # dispersion spreading of that signal bandwidth
    disp_ps   = D_ps_nm_km * L_km * BW_nm     # ps of broadening
    fits_grid = "YES" if BW_ghz < 100 else "NO (wider)"
    needs_dsp = "YES (%.0f ps spread)" % disp_ps if disp_ps > T_sym_ps/4 else "no"
    print(f"  {rate_gbps:>4d} Gbps  {T_sym_ps:>6.1f}ps  {BW_ghz:>10.1f}GHz  {disp_ps:>10.1f}ps  {fits_grid:>14s}  {needs_dsp}")
print()
print("  -> All rates fit 100 GHz ITU spacing (signal BW << 100 GHz)")
print("  -> Dispersion compensation (DCF or DSP) needed at 100G+ over 100 km")